# Nomobug Prospects: 2026 and B2B Follow Up

Beginner-friendly CP2 notebook. Run the cells from top to bottom.

It loads **all populated rows** but only displays small previews so the notebook stays responsive. Credentials and real source IDs stay in the ignored local `secrets/` and `data/profiles/` folders.

## 1. Import pandas and connect to the existing local Google OAuth files

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

token_path = project_root / "secrets" / "google_token.json"
metadata_path = project_root / "data" / "profiles" / "google_source_metadata.json"

print("Project folder:", project_root)
print("Token found:", token_path.exists())
print("Source metadata found:", metadata_path.exists())

In [ ]:
scopes = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/calendar.readonly",
]

credentials = Credentials.from_authorized_user_file(str(token_path), scopes)
sheets_service = build("sheets", "v4", credentials=credentials, cache_discovery=False)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print("Google Sheets connection ready")

## 2. Find the Prospects workbook

The spreadsheet ID is read from ignored metadata and is never written into this notebook.

In [ ]:
workbook_title = "Prospects List - Nomobug"
spreadsheet_id = ""

for workbook in metadata["spreadsheets"]:
    if workbook["properties"]["title"] == workbook_title:
        spreadsheet_id = workbook["spreadsheetId"]

print("Workbook found:", bool(spreadsheet_id))

## 3. Load every populated row from the 2026 tab

In [ ]:
response_2026 = sheets_service.spreadsheets().values().get(
    spreadsheetId=spreadsheet_id,
    range="'2026'",
    majorDimension="ROWS",
).execute()

rows_2026 = response_2026.get("values", [])
width_2026 = max(len(row) for row in rows_2026)
columns_2026 = rows_2026[0] + [""] * (width_2026 - len(rows_2026[0]))
columns_2026 = [name.strip() or f"unnamed_{i + 1}" for i, name in enumerate(columns_2026)]
data_2026 = [(row + [""] * width_2026)[:width_2026] for row in rows_2026[1:]]

df_2026 = pd.DataFrame(data_2026, columns=columns_2026)
df_2026 = df_2026.replace(r"^\s*$", pd.NA, regex=True).dropna(how="all")

print("All loaded rows:", len(df_2026))
print("Columns:", len(df_2026.columns))
display(df_2026.head())

## 4. Check quality without displaying customer phone numbers

Exact duplicate rows and repeat phone interactions are different concepts. A repeat phone can be a real new advertisement conversation.

In [ ]:
profile_2026 = pd.DataFrame({
    "column": df_2026.columns,
    "non_null_count": df_2026.notna().sum().values,
    "missing_count": df_2026.isna().sum().values,
    "missing_percent": (df_2026.isna().mean() * 100).round(2).values,
    "distinct_count": df_2026.nunique(dropna=True).values,
})

print("Exact duplicate rows:", int(df_2026.duplicated().sum()))
print("These exact duplicates need investigation before removal.")
display(profile_2026)

In [ ]:
phone_clean = (
    df_2026["NO TELEFON"]
    .astype("string")
    .str.replace(r"[^0-9]", "", regex=True)
)

print("Rows with a phone:", int(phone_clean.notna().sum()))
print("Distinct normalized phones:", int(phone_clean.nunique(dropna=True)))
print("Rows sharing a phone with another interaction:", int(phone_clean.duplicated(keep=False).sum()))
print("No phone values are displayed here.")

## 5. Load every populated row from B2B FOLLOW UP

In [ ]:
response_b2b = sheets_service.spreadsheets().values().get(
    spreadsheetId=spreadsheet_id,
    range="'B2B FOLLOW UP'",
    majorDimension="ROWS",
).execute()

rows_b2b = response_b2b.get("values", [])
width_b2b = max(len(row) for row in rows_b2b)
columns_b2b = rows_b2b[0] + [""] * (width_b2b - len(rows_b2b[0]))
columns_b2b = [name.strip() or f"unnamed_{i + 1}" for i, name in enumerate(columns_b2b)]
data_b2b = [(row + [""] * width_b2b)[:width_b2b] for row in rows_b2b[1:]]

df_b2b = pd.DataFrame(data_b2b, columns=columns_b2b)
df_b2b = df_b2b.replace(r"^\s*$", pd.NA, regex=True).dropna(how="all")

print("All loaded rows:", len(df_b2b))
print("Exact duplicate rows:", int(df_b2b.duplicated().sum()))
display(df_b2b.head())

## 6. Save every loaded row locally

The CSV files are ignored by Git and will not be uploaded to GitHub.

In [ ]:
raw_folder = project_root / "data" / "raw" / "prospects"
profile_folder = project_root / "data" / "profiles" / "prospects"
raw_folder.mkdir(parents=True, exist_ok=True)
profile_folder.mkdir(parents=True, exist_ok=True)

df_2026.to_csv(raw_folder / "2026.csv", index=False)
df_b2b.to_csv(raw_folder / "B2B_FOLLOW_UP.csv", index=False)
profile_2026.to_csv(profile_folder / "2026_column_profile.csv", index=False)

print("Saved all rows to:", raw_folder)
print("Saved profile to:", profile_folder)